# 2. Cleaning and Preprocessing

In this section, we prepare the data for modelling based on the issues identified during EDA.

We will handle missing values and unusual observations, prepare categorical variables, explore possible feature engineering and apply scaling where needed.

The goal is to create a clean and consistent dataset that can be used to train our classification model.

## Import Libraries

In [155]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 2.1 Loading the Data

We start by loading the original datasets again. We keep the original data unchanged so that we can compare the results before and after preprocessing.

In [156]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
clientes = pd.read_csv("../data/clientes.csv")

# Create a copy of the training data
train_clean = train.copy()

train_clean.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.924510
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.258730
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448402
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.058890


## 2.2 Handling Missing Values

During EDA, we found missing values in several variables, especially those related to previous purchases.

We will handle missing values differently depending on their possible causes, instead of applying the same method to every variable.

### 2.2.1 Handling Missing Values Due to Limited Purchase History

We created a reusable function to handle missing values caused by insufficient purchase history. This includes undefined standard deviations and other historical variables that cannot be calculated when customers have very few previous purchases.

The same function will also be applied to the test data to ensure consistent cleaning.

In [157]:
# • Handle Missing Values Due to Limited Purchase History

def clean_missing_history(df):
    df = df.copy()

    # Record missing standard deviations
    df["std_purchase_missing"] = df["prior_std_purchase_value"].isna().astype(int)
    df["std_gap_missing"] = df["prior_std_gap_days"].isna().astype(int)

    # Fill undefined standard deviations with zero
    df.loc[
        (df["prior_purchase_count"] < 2) &
        df["prior_std_purchase_value"].isna(),
        "prior_std_purchase_value"
    ] = 0

    df.loc[
        (df["prior_purchase_count"] < 3) &
        df["prior_std_gap_days"].isna(),
        "prior_std_gap_days"
    ] = 0

    # Handle customers with no previous purchases
    no_history = df["prior_purchase_count"] == 0
    df["no_previous_purchases"] = no_history.astype(int)

    history_cols = [
        "days_since_previous_purchase",
        "prior_mean_purchase_value",
        "prior_min_purchase_value",
        "prior_max_purchase_value"
    ]

    df.loc[no_history, history_cols] = (
        df.loc[no_history, history_cols].fillna(0)
    )

    # Handle insufficient history for average purchase gaps
    limited_history = df["prior_purchase_count"] < 2
    df["insufficient_gap_history"] = limited_history.astype(int)

    df.loc[
        limited_history & df["prior_mean_gap_days"].isna(),
        "prior_mean_gap_days"
    ] = 0

    return df


# Apply cleaning to training data
train_clean = clean_missing_history(train)

In [158]:
columns = [
    "prior_std_purchase_value",
    "prior_std_gap_days"
]

comparison = pd.DataFrame({
    "Before": train[columns].isna().sum(),
    "After": train_clean[columns].isna().sum()
})

comparison

,Before,After
prior_std_purchase_value,1113,65
prior_std_gap_days,1478,65


The number of missing values decreased significantly after applying the cleaning rules. The remaining 65 missing values correspond to observations with almost no transaction information, which we investigate separately.

### 2.2.2 Verifying Other Historical Missing-Value Corrections

We now compare missing values before and after cleaning for the remaining historical purchase variables, to verify that our corrections worked.

In [175]:
columns = [
    "days_since_previous_purchase",
    "prior_mean_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "prior_mean_gap_days"
]

comparison = pd.DataFrame({
    "Before": train[columns].isna().sum(),
    "After": train_clean[columns].isna().sum()
})

comparison

,Before,After
days_since_previous_purchase,861,274
prior_mean_purchase_value,862,65
prior_min_purchase_value,668,65
prior_max_purchase_value,668,65
prior_mean_gap_days,1308,276


We replaced missing historical values caused by limited purchase history with zero, while keeping indicators to identify these cases.

Some missing values still remain and will require a different approach during preprocessing.

### 2.2.3 Extremely Incomplete Observations

During EDA, we identified 65 observations with 36 missing values each. We now investigate what information is still available in these observations before deciding whether to keep or remove them.

In [160]:
incomplete_rows = train[train.isna().sum(axis=1) >= 30]

print("Number of incomplete observations:", len(incomplete_rows))

print("\nAvailable information:")
print(incomplete_rows.notna().sum()[lambda x: x > 0])

Number of incomplete observations: 65

Available information:
repeat_purchase_90d    65
customer_id            65
ID                     65
dtype: int64


In [161]:
# • Check Missing Purchase Dates

print("Missing purchase dates:", train["purchase_date"].isna().sum())

print(
    "Missing dates in incomplete observations:",
    incomplete_rows["purchase_date"].isna().sum()
)

Missing purchase dates: 65
Missing dates in incomplete observations: 65


We identified 65 observations with almost all transaction information missing, including `purchase_date`. Although these observations can be matched with `clientes.csv`, the additional customer information cannot recover the missing transaction details.

We exclude these observations from our chronological training and validation split because their purchase dates are unavailable.

## 2.3 Handling Anomalous and Implausible Values

During EDA, we identified negative time intervals, negative monetary values and extremely large amounts.

We will investigate these values and correct those that appear to be errors, while keeping as much useful information as possible.

#### • Investigate Negative Annual Spending

In [162]:
negative_spend = train[train["spend_365d"] < 0]

display(negative_spend[[
    "ID",
    "prior_total_spend",
    "spend_180d",
    "spend_365d"
]])

,ID,prior_total_spend,spend_180d,spend_365d
263,10000300,3456.32,577.67,-3456.320000
1733,10002022,5129.20,264.00,-1230.720000
1744,10002033,2586.88,0.00,-167.180000
1829,10002134,753.51,0.00,-753.510000
2099,10002452,28625.97,2234.61,-2648.440000
2851,10003340,31123.69,0.00,-2169.910000
3686,10004321,3215.36,420.49,-1443.640000
3690,10004327,9615.39,591.02,-5133.040000
3841,10004511,58473.69,9560.84,-14740.290000
3893,10004574,10042.78,179.58,-4196.900000


#### • Correct Anomalous Values

In [163]:
def clean_anomalies(df):
    df = df.copy()

    # Recalculate average purchase values
    expected = (
        df["prior_total_spend"] /
        df["prior_purchase_count"].replace(0, np.nan)
    )

    valid = expected.notna()
    df.loc[valid, "prior_mean_purchase_value"] = expected[valid]

    # Identify negative annual spending
    previous_flag = df.get("spend_365d_negative", 0)

    df["spend_365d_negative"] = (
        (previous_flag == 1) | (df["spend_365d"] < 0)
    ).astype(int)

    # Replace negative time intervals and annual spending with NaN
    for col in [
        "days_since_previous_purchase",
        "prior_mean_gap_days",
        "spend_365d"
    ]:
        df.loc[df[col] < 0, col] = np.nan

    # Replace suspicious values of 1 billion
    for col in ["prior_mean_purchase_value", "spend_365d"]:
        df.loc[df[col] == 1_000_000_000, col] = np.nan

    return df

#### • Apply Anomaly Cleaning

In [164]:
train_clean = clean_anomalies(train_clean)

#### • Verify Anomaly Corrections

In [165]:
expected = (
    train_clean["prior_total_spend"] /
    train_clean["prior_purchase_count"].replace(0, np.nan)
)

valid = expected.notna()

print(
    "Remaining inconsistent averages:",
    (valid & ~np.isclose(
        train_clean["prior_mean_purchase_value"], expected
    )).sum()
)

Remaining inconsistent averages: 0


We recalculated average purchase values using total previous spending and purchase count, correcting inconsistent and extreme values where possible.

Negative time intervals and suspicious annual spending values were replaced with NaN. We kept the observations to preserve their remaining information.

## 2.4 Customer Data Integration

We decided to add information from `clientes.csv` to our dataset. We selected `commercial_region` and `customer_since_year` because they might help explain customer purchasing behaviour.

#### • Integrate Customer Information

In [166]:
customer_info = clientes[
    ["customer_id", "commercial_region", "customer_since_year"]
]

# Create a reusable function
def merge_customer_info(df):
    return df.merge(
        customer_info,
        on="customer_id",
        how="left",
        validate="many_to_one"
    )

# Apply it to training data
train_clean = merge_customer_info(train_clean)

# Preview the result
train_clean[[
    "customer_id",
    "commercial_region",
    "customer_since_year"
]].head()

,customer_id,commercial_region,customer_since_year
0,660784,Region RG25: We Will See Later,2024
1,189146,Region RG17: We Will See Later,2015
2,103940,Region RG23: We Will See Later,2017
3,512927,Region RG17: We Will See Later,2018
4,122356,Region RG31: We Will See Later,2017


#### • Verify Customer Data Integration

In [167]:
print("Number of rows:", len(train_clean))

print("\nMissing values:")
print(train_clean[
    ["commercial_region", "customer_since_year"]
].isna().sum())

Number of rows: 6534

Missing values:
commercial_region      35
customer_since_year     0
dtype: int64


We selected `commercial_region` and `customer_since_year` because they may provide useful information about purchasing behaviour.

We excluded the other customer variables because some contain many missing values, while others, such as company names and addresses, are too specific to individual customers.

## 2.5 Feature Engineering

We created additional features to better describe customer purchasing behaviour.

From `purchase_date`, we extracted the month and weekday to capture possible seasonal patterns. We treat these as categorical variables because they do not have a meaningful numerical order.

In [168]:
# • Create Date Features

def add_date_features(df):
    df = df.copy()

    df["purchase_date"] = pd.to_datetime(
        df["purchase_date"], errors="coerce"
    )

    df["purchase_month"] = (
        df["purchase_date"].dt.month.astype("Int64").astype(str)
    )

    df["purchase_weekday"] = (
        df["purchase_date"].dt.dayofweek.astype("Int64").astype(str)
    )

    return df


train_clean = add_date_features(train_clean)

train_clean[[
    "purchase_date",
    "purchase_month",
    "purchase_weekday"
]].head()

,purchase_date,purchase_month,purchase_weekday
0,2024-10-29,10,1
1,2017-07-08,7,5
2,2019-03-19,3,1
3,2018-11-10,11,5
4,2021-11-30,11,1


We also created two features to better describe purchasing behaviour.

`purchase_gap_ratio` compares the time since the previous purchase with the customer's usual purchase interval.

`recent_spend_share` measures the proportion of annual spending made in the last 30 days.

These features may help identify changes in customer purchasing patterns.

In [169]:
# • Create Additional Purchase Features

def add_purchase_features(df):
    df = df.copy()

    # Compare current purchase gap with usual purchase gap
    df["purchase_gap_ratio"] = (
        df["days_since_previous_purchase"] /
        df["prior_mean_gap_days"].replace(0, np.nan)
    )

    # Calculate share of annual spending in the last 30 days
    df["recent_spend_share"] = (
        df["spend_30d"] /
        df["spend_365d"].replace(0, np.nan)
    )

    return df


train_clean = add_purchase_features(train_clean)

## 2.6 Training and Validation Split

We used a chronological split, keeping earlier purchases for training and more recent purchases for validation. We also left a 90-day gap between the two sets to reduce the risk of data leakage, since our target predicts purchases within 90 days.

#### • Split Training and Validation Data

In [170]:
data = train_clean.dropna(subset=["purchase_date"]).copy()

# Use the most recent 20% of dated observations for validation
cutoff = data["purchase_date"].quantile(0.80)

train_part = data[
    data["purchase_date"] < cutoff - pd.Timedelta(days=90)
]

val_part = data[
    data["purchase_date"] >= cutoff
]

# Separate the target from the features
drop_cols = [
    "repeat_purchase_90d",
    "ID",
    "customer_id",
    "purchase_date",
    "random_noise"
]

X_train = train_part.drop(columns=drop_cols)
X_val = val_part.drop(columns=drop_cols)

y_train = train_part["repeat_purchase_90d"]
y_val = val_part["repeat_purchase_90d"]

print("Training observations:", len(X_train))
print("Validation observations:", len(X_val))

Training observations: 5024
Validation observations: 1298


We excluded `ID` and `customer_id` because they are identifiers rather than measures of purchasing behaviour. We also removed `random_noise`, since it does not contain meaningful information for prediction.

## 2.7 Preprocessing Pipeline

We created a preprocessing pipeline to handle the remaining missing values, categorical encoding and numerical scaling.

Numerical missing values are replaced using the median, while categorical missing values are assigned a separate "Missing" category.

We use One-Hot Encoding because our categorical variables have no natural order. Rare categories appearing fewer than 10 times in the training data are grouped together to reduce the number of features and avoid creating separate columns for very uncommon categories.

All transformations are fitted only on the training data to avoid data leakage.
We use StandardScaler to put numerical features on comparable scales, which is particularly useful for scale-sensitive models such as Logistic Regression.

In [171]:
# • Create Preprocessing Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# • Handle Rare Categories with One-Hot Encoding
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=10
    ))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])

# Fit on training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same transformations to validation data
X_val_processed = preprocessor.transform(X_val)

print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)

Training shape: (5024, 137)
Validation shape: (1298, 137)


## 2.8 Preparing the Test Data

We apply the same preprocessing steps to `test.csv`, including missing-value treatment, anomaly correction, customer data integration and feature engineering.

We then use the previously fitted preprocessing pipeline to transform the test data without fitting it again, avoiding data leakage.

#### • Handle Missing Values in Test Data

In [172]:
test_clean = clean_missing_history(test)

#### • Apply Anomaly Corrections

In [173]:
# • Apply Anomaly Corrections

test_clean = clean_anomalies(test_clean)

# • Integrate Customer Information

test_clean = merge_customer_info(test_clean)

# • Create Date Features

test_clean = add_date_features(test_clean)

# • Create Additional Purchase Features

test_clean = add_purchase_features(test_clean)

We applied the same missing-value and anomaly corrections used for the training data. We also merged the selected customer information and created the same date features to maintain consistency between the datasets.

#### • Apply Preprocessing Pipeline to Test Data

In [176]:
# • Apply Preprocessing Pipeline to Test Data

# Remove columns not used for modelling
X_test = test_clean.drop(columns=drop_cols, errors="ignore")

# • Check Missing Features in Test Data

missing_cols = X_train.columns.difference(X_test.columns)

print(missing_cols.tolist())

# Keep the same feature columns as the training data
X_test = X_test[X_train.columns]

# Apply the previously fitted preprocessing pipeline
X_test_processed = preprocessor.transform(X_test)

# Verify the results
print("Test shape:", X_test_processed.shape)
print("Same number of features:",
      X_test_processed.shape[1] == X_train_processed.shape[1])
print("All rows preserved:", len(test_clean) == len(test))
print("IDs preserved:", test_clean["ID"].equals(test["ID"]))

[]
Test shape: (986, 137)
Same number of features: True
All rows preserved: True
IDs preserved: True


The test data was prepared using the same preprocessing rules as the training data.

The final dataset contains 986 observations and 137 features, matching the training features. All observations and original IDs were preserved.

The preprocessing pipeline was applied without fitting it again on the test data, avoiding data leakage.